In [1]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, joblib, time
from sklearn.svm import OneClassSVM

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.features import to_matrix, ClippedScaler    # needed to load the saved scaler
from src.evaluation import evaluate
from src.pipeline import DATA, RESULTS, score_split

SEED = 11
d = joblib.load(RESULTS / "scaler_v2.joblib")
cols, scaler = d["cols"], d["scaler"]
print(len(cols), "features")

28 features


In [2]:
tr = pd.read_parquet(DATA / "train.parquet", columns=cols).sample(20_000, random_state=SEED)
x_train = scaler.transform(to_matrix(tr, cols)).astype(np.float32)

t0 = time.time()
model = OneClassSVM(kernel="rbf", gamma="scale", nu=0.05).fit(x_train)
print("fit seconds:", round(time.time() - t0), "| support vectors:", len(model.support_))

fit seconds: 3 | support vectors: 1018


In [3]:
def svm_score(x, chunk=20_000):
    out = []
    for i in range(0, len(x), chunk):
        out.append(-model.decision_function(x[i:i + chunk]))   # higher = more anomalous
    return np.concatenate(out)

t0 = time.time()
val_s = score_split("val", cols, scaler, svm_score)
tune_s = score_split("tune", cols, scaler, svm_score)
test_s = score_split("test", cols, scaler, svm_score, cap=100_000)
print("scoring minutes:", round((time.time() - t0) / 60, 1))

keys = ["auroc", "tpr@fpr0.01", "real_fpr@0.01", "tpr@fpr0.05", "macro_tpr@fpr0.05", "real_fpr@0.05"]
for name, d_ in [("tune", tune_s), ("test", test_s)]:
    r = evaluate(d_, val_df=val_s)
    o = evaluate(d_)
    print(name, "val-threshold:", {k: round(r[k], 3) for k in keys})
    print(name, "ORACLE tpr@1%:", round(o["tpr@fpr0.01"], 3))

pd.concat([val_s, tune_s, test_s]).to_parquet(
    RESULTS / "scores" / f"m2_ocsvm_nftoniot_seed{SEED}.parquet")

scoring minutes: 7.7
tune val-threshold: {'auroc': 0.868, 'tpr@fpr0.01': 0.004, 'real_fpr@0.01': 0.015, 'tpr@fpr0.05': 0.046, 'macro_tpr@fpr0.05': 0.531, 'real_fpr@0.05': 0.051}
tune ORACLE tpr@1%: 0.002
test val-threshold: {'auroc': 0.391, 'tpr@fpr0.01': 0.281, 'real_fpr@0.01': 0.422, 'tpr@fpr0.05': 0.623, 'macro_tpr@fpr0.05': 0.659, 'real_fpr@0.05': 0.77}
test ORACLE tpr@1%: 0.008
